# debugging the localizer loop

**Generate** draws one question the way `s2_pretraining_localizer` draws one slot of an image. It does not run the Localizer, and it does not load the Gemma weights. The board is a `new_multi_gold_game` with `n_gold` uniform in 0–3, opening `any`, and `require_target`. The picture is that frame after `noise_image` at `TRAINING_STRENGTH`. The noise does not move the labels.

The beginning coin is the box **Begin fraction** (the trainer default is 0.25). A beginning uses `Trainer._begin_targets`: a look and a move for each gold, or for each exit when there is no gold, stopping at the trainer's six-question cap. This button shows one of those replies and does not pad the batch. A later image uses `board_context` and one `draw_questions` call.

Reply tokens come from `Collator.build`: the teacher-forced reply plus the terminator from `resolve_terminator_id`. Click one. Blue is **s**, red is **v**, orange is **v_bar**, green is the average of **v** and **v_bar**. On a full token those four are the stored label. On a beginning, tokens through the `]` of `[REMEMBER ...]` are midpoint tokens: **s** and the green average are the agent, and **v** and **v_bar** are not supervised. Dotted underlines are midpoint tokens.

The ratios are the images drawn since **Clear**. Beginning versus later should track the fraction in the box. Later images carry a target note with probability 0.75. Among questions on those images, half ask for your target. Look versus move is a coin of 0.5, including the one beginning reply this button keeps. Change the fraction, then **Clear**, before reading the ratio against the new coin.

Run this on the GPU box. The first **Generate** loads the Gemma processor, which is the tokenizer the trainer's collator uses. The cell needs the repo environment (`pygame`, `torch`, `ipywidgets`).



In [ ]:
from __future__ import annotations

import html
import io
import math
import os
import random
import sys
import tempfile
import traceback

os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

_here_name = os.path.basename(os.getcwd())
if _here_name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import ipywidgets as widgets
import numpy as np
from IPython.display import HTML, display
from PIL import Image

from agent.config import CONFIG
from agent.memory import format_notepad
from agent.model import ADAPTERS, spec_for
from agent.modes import (
    S2_BEGINNING_USER,
    SYSTEM_PROMPT_S2,
    _S2_LOOK_LINES,
    _S2_MOVE_LINES,
    _build_game_messages,
)
from neural_net.s2_pretraining_learn_to_look import (
    BoardPump,
    _coords,
    _open_png,
    beginning_messages,
    board_context,
    draw_questions,
)
from neural_net.s2_pretraining_localizer import (
    Trainer,
    _begin_reply,
    _close_content_index,
)
from training.image_noise import TRAINING_STRENGTH, noise_image
from training.train import Collator, TrainingExample, resolve_terminator_id

BEGIN_FRACTION = 0.25
N_GOLD = (0, 1, 2, 3)
BOARD_PX = 560
_RADIUS_ONE = 8
_RADIUS_ALL = 3
_COLOR = {
    "s": (30, 90, 255),
    "v": (255, 0, 0),
    "v_bar": (255, 140, 0),
    "avg": (0, 200, 70),
}
_DRAW_ORDER = ("s", "v_bar", "v", "avg")
_FRAME_PATH = os.path.join(tempfile.gettempdir(), "debugging_localizer_loop.png")

rng = random.Random()
pump = BoardPump(0, CONFIG.game_size, N_GOLD, rng, require_target=True)

tokens = []
tally = {
    "images": 0,
    "beginning": 0,
    "later": 0,
    "later_note": 0,
    "look": 0,
    "move": 0,
    "your_target": 0,
    "explicit": 0,
    "noted_q": 0,
}
state = {
    "busy": False,
    "ignore_through": 0,
    "gen": 0,
    "frame": None,
    "selected": None,
    "collator": None,
}


def _say(text):
    log.append_stdout(text if text.endswith("\n") else text + "\n")


def _set_busy(flag):
    state["busy"] = flag
    for widget in controls:
        widget.disabled = flag


def _collator():
    """The trainer's processor, terminator, and collator. No model weights."""
    if state["collator"] is not None:
        return state["collator"]
    import torch
    from transformers import AutoProcessor

    spec = spec_for("gemma-4-12b")
    processor = AutoProcessor.from_pretrained(
        spec.hf_id,
        token=CONFIG.hf_token,
        trust_remote_code=spec.trust_remote_code,
    )
    tokenizer = getattr(processor, "tokenizer", processor)
    terminator = resolve_terminator_id(None, tokenizer)
    collator = Collator(
        processor,
        ADAPTERS[spec.family],
        terminator,
        compute_dtype=torch.float32,
        device="cpu",
    )
    state["collator"] = collator
    return collator


def _on_board(x, y):
    return (
        math.isfinite(x) and math.isfinite(y)
        and 0.0 <= x <= 1.0 and 0.0 <= y <= 1.0
    )


def _blit_dot(out, x, y, color, radius):
    if x is None or y is None or not _on_board(x, y):
        return
    height, width = out.shape[:2]
    col = int(round(float(x) * (width - 1)))
    row = int(round((1.0 - float(y)) * (height - 1)))
    y0 = max(0, row - radius)
    y1 = min(height, row + radius + 1)
    x0 = max(0, col - radius)
    x1 = min(width, col + radius + 1)
    yy, xx = np.ogrid[y0:y1, x0:x1]
    mask = (yy - row) ** 2 + (xx - col) ** 2 <= radius ** 2
    out[y0:y1, x0:x1][mask] = color


def _paint(out, rows, radius):
    for key in _DRAW_ORDER:
        color = _COLOR[key]
        for row in rows:
            point = row[key]
            if point is None:
                continue
            _blit_dot(out, point[0], point[1], color, radius)


def _png(rgb):
    buf = io.BytesIO()
    Image.fromarray(np.ascontiguousarray(rgb), mode="RGB").save(buf, format="PNG")
    return buf.getvalue()


def _show_board():
    frame = state["frame"]
    if frame is None:
        return
    selected = state["selected"]
    show = bool(show_all.value) and bool(tokens)
    one = (
        tokens[selected]
        if selected is not None and 0 <= selected < len(tokens)
        else None
    )
    if not show and one is None:
        board.value = _png(frame)
        return
    out = np.array(frame, copy=True)
    if show:
        _paint(out, tokens, _RADIUS_ALL)
    if one is not None:
        _paint(out, [one], _RADIUS_ONE)
    board.value = _png(out)


def _visible(piece):
    if piece == "":
        return "∅"
    return piece.replace(" ", "\u00a0")


def _token_markup():
    if not tokens:
        return "<div class='dbg-tokens'>Generate to get the reply tokens.</div>"
    gen = state["gen"]
    selected = state["selected"]
    parts = []
    current = None
    for row in tokens:
        if row["q"] != current:
            if current is not None:
                parts.append("</div>")
            current = row["q"]
            parts.append(
                "<div class='dbg-q'><div class='dbg-h'>"
                f"question {row['q']} · {html.escape(row['kind'])} · "
                f"{html.escape(row['phrase'])} · {html.escape(row['question_kind'])}"
                "</div>"
                f"<div class='dbg-line'>{html.escape(row['question'])}</div>"
                f"<div class='dbg-line'>reply: {html.escape(row['reply'])}</div>"
                f"<div class='dbg-line'>{html.escape(row['label_text'])}</div>"
                "<div class='dbg-toks'>"
            )
        cls = "dbg-tok"
        if row["eos"]:
            cls += " dbg-eos"
        if row["mode"] == "mid":
            cls += " dbg-mid"
        if row["i"] == selected:
            cls += " dbg-on"
        parts.append(
            f"<span class='{cls}' data-gen='{gen}' data-i='{row['i']}'>"
            f"{html.escape(_visible(row['piece']))}</span>"
        )
    if current is not None:
        parts.append("</div></div>")
    return "<div class='dbg-tokens'>" + "".join(parts) + "</div>"


def _show_tokens():
    token_html.value = _token_markup()


def _point_text(name, xy):
    if xy is None:
        return f"{name}=unsupervised"
    x, y = xy
    extra = "" if _on_board(x, y) else " (off the board)"
    return f"{name}=({x:.4f}, {y:.4f}){extra}"


def _fmt(coords):
    sx, sy, vx, vy, bx, by = coords
    return (
        f"stored label  s=({sx:.4f}, {sy:.4f})  "
        f"v=({vx:.4f}, {vy:.4f})  v_bar=({bx:.4f}, {by:.4f})"
    )


def _ratio_line(name, num, den, expected):
    if den <= 0:
        got = "—"
    else:
        got = f"{num / den:.3f}"
    return (
        f"{html.escape(name)}: {num}/{den} = {got} "
        f"(coin {expected:.2f})"
    )


def _show_ratios():
    n = tally["images"]
    later = tally["later"]
    frac = float(begin_box.value)
    lines = [
        _ratio_line("beginning images", tally["beginning"], n, frac),
        _ratio_line("later images", later, n, 1.0 - frac),
        _ratio_line(
            "later images with a target note", tally["later_note"], later, 0.75,
        ),
        _ratio_line(
            "your-target among noted questions",
            tally["your_target"], tally["noted_q"], 0.5,
        ),
        _ratio_line(
            "look among questions",
            tally["look"], tally["look"] + tally["move"], 0.5,
        ),
    ]
    body = "<br>".join(lines)
    ratios.value = (
        "<div style='font-family:monospace;font-size:13px;color:#222'>"
        f"images {n}<br>{body}</div>"
    )


def _messages_text(messages):
    chunks = []
    for message in messages:
        role = message.get("role", "?")
        content = message.get("content")
        bits = []
        if isinstance(content, str):
            bits.append(content)
        elif isinstance(content, list):
            for part in content:
                if not isinstance(part, dict):
                    continue
                if part.get("type") == "image":
                    bits.append("[image: noised training frame]")
                elif part.get("type") == "text":
                    bits.append(part.get("text") or "")
        chunks.append(f"--- {role} ---\n" + "\n".join(bits))
    return "\n\n".join(chunks)


def _reply_tokens(question, q_index, collator):
    built = collator.build(TrainingExample(
        messages=question["messages"],
        target_text=question["reply"],
        loss="ce",
        source="s2_localizer",
        meta={"coords": list(question["coords"])},
    ))
    weights = built["weights"][0]
    ids = built["model_inputs"]["input_ids"][0]
    reply_pos = (weights != 0).nonzero(as_tuple=False).flatten()
    if int(reply_pos.numel()) == 0:
        raise RuntimeError("question has no reply tokens")
    tokenizer = collator.tokenizer
    close = None
    if question["question_kind"] == "beginning":
        close = _close_content_index(tokenizer, question["reply"])
    coords = question["coords"]
    sx, sy, vx, vy, bx, by = coords
    agent = (sx, sy)
    full_v = (vx, vy)
    full_bar = (bx, by)
    full_avg = ((vx + bx) / 2.0, (vy + by) / 2.0)
    rows = []
    for j, pos in enumerate(reply_pos.tolist()):
        # Same split as the trainer: a beginning reply supervises the
        # midpoint through the token that contains ']'. The terminator
        # and every later token are a full look or move.
        tid = int(ids[pos])
        mid = close is not None and j <= close
        piece = tokenizer.decode([tid], skip_special_tokens=False)
        if mid:
            s_pt, v_pt, bar_pt, avg_pt = agent, None, None, agent
            mode = "mid"
        else:
            s_pt, v_pt, bar_pt, avg_pt = agent, full_v, full_bar, full_avg
            mode = "full"
        rows.append({
            "i": len(tokens) + len(rows),
            "q": q_index,
            "id": int(tid),
            "piece": piece,
            "eos": j == int(reply_pos.numel()) - 1,
            "mode": mode,
            "kind": question["kind"],
            "phrase": question["phrase"],
            "question_kind": question["question_kind"],
            "question": question["question"],
            "reply": question["reply"],
            "label_text": _fmt(coords),
            "s": s_pt,
            "v": v_pt,
            "v_bar": bar_pt,
            "avg": avg_pt,
        })
    return rows


def _one_question(kind, phrase, point, candidate_kind, question_kind,
                  question, coords, reply, messages, notepad, context):
    return {
        "kind": kind,
        "phrase": phrase,
        "point": point,
        "candidate_kind": candidate_kind,
        "question_kind": question_kind,
        "question": question,
        "coords": coords,
        "reply": reply,
        "messages": messages,
        "notepad": notepad,
        "context": context,
    }


def _sample(begin_p):
    raw = pump.take()
    settings = raw["settings"]
    noised = noise_image(_open_png(raw["png"]), rng, TRAINING_STRENGTH).convert("RGB")
    noised.save(_FRAME_PATH, format="PNG")
    frame = np.asarray(noised)
    begin = rng.random() < begin_p
    agent = (float(settings["agent_x"]), float(settings["agent_y"]))
    if begin:
        notepad = format_notepad([])
        messages = beginning_messages(_FRAME_PATH, notepad)
        targets = Trainer._begin_targets(None, settings)
        # Same stop as Trainer._encode_begin at the default of six
        # questions. This view keeps one of those replies and does not
        # pad the rest.
        built = []
        for cand_kind, phrase, point in targets:
            if len(built) + 2 > 6:
                break
            for kind in ("look", "move"):
                built.append(_one_question(
                    kind, phrase, point, cand_kind, "beginning",
                    S2_BEGINNING_USER, _coords(kind, agent, point),
                    _begin_reply(phrase, kind), messages, notepad, "",
                ))
        if len(built) < 2:
            raise RuntimeError(
                f"a beginning needs a look and a move, got {len(built)}"
            )
        question = rng.choice(built)
        note = None
    else:
        ctx = board_context(settings, rng)
        drawn = draw_questions(ctx, rng, 1)
        if not drawn:
            raise RuntimeError("draw_questions returned nothing")
        item = drawn[0]
        messages = _build_game_messages(
            SYSTEM_PROMPT_S2, _FRAME_PATH, ctx.context, item.question,
            notepad=ctx.notepad,
        )
        reply = rng.choice(
            _S2_LOOK_LINES if item.kind == "look" else _S2_MOVE_LINES
        )
        question = _one_question(
            item.kind, item.phrase, item.point, item.candidate_kind,
            item.question_kind, item.question, item.coords, reply,
            messages, ctx.notepad, ctx.context,
        )
        note = ctx.target_note
    return {
        "begin": begin,
        "settings": settings,
        "frame": frame,
        "question": question,
        "note": note,
        "agent": agent,
    }


def _count(sample):
    tally["images"] += 1
    noted = sample["note"] is not None
    if sample["begin"]:
        tally["beginning"] += 1
    else:
        tally["later"] += 1
        if noted:
            tally["later_note"] += 1
    question = sample["question"]
    if question["kind"] == "look":
        tally["look"] += 1
    else:
        tally["move"] += 1
    if sample["begin"]:
        return
    if noted:
        tally["noted_q"] += 1
    if question["question_kind"] == "your_target":
        tally["your_target"] += 1
    else:
        tally["explicit"] += 1


def on_generate(_):
    if state["busy"]:
        return
    try:
        begin_p = float(begin_box.value)
    except (TypeError, ValueError) as exc:
        status.value = str(exc)
        return
    if not 0.0 <= begin_p <= 1.0:
        status.value = "The beginning fraction must be between 0 and 1."
        return
    _set_busy(True)
    status.value = "Drawing one training question."
    try:
        collator = _collator()
        sample = _sample(begin_p)
        _count(sample)
        tokens.clear()
        state["selected"] = None
        state["gen"] += 1
        state["frame"] = sample["frame"]
        question = sample["question"]
        tokens.extend(_reply_tokens(question, 0, collator))
        style = "beginning" if sample["begin"] else "later"
        settings = sample["settings"]
        golds = settings.get("gold") or []
        openings = settings.get("openings") or []
        note = sample["note"] if sample["note"] is not None else "(none)"
        _say(
            f"{style}  golds={len(golds)}  openings={len(openings)}  "
            f"note={note}  {question['kind']}  {question['phrase']}  "
            f"agent=({sample['agent'][0]:.4f}, {sample['agent'][1]:.4f})"
        )
        _say(_messages_text(question["messages"]))
        _say(f"reply: {question['reply']}")
        _say("")
        _show_ratios()
        _show_tokens()
        _show_board()
        status.value = (
            "Click a reply token. Blue is s, red is v, orange is v_bar, "
            "green is the average of v and v_bar. A midpoint token draws "
            "s and that average on the agent; v and v_bar are not supervised."
        )
    except Exception:
        status.value = "Generate failed."
        _say(traceback.format_exc())
        raise
    finally:
        _set_busy(False)


def on_clear(_):
    if state["busy"]:
        return
    for key in tally:
        tally[key] = 0
    tokens.clear()
    state["selected"] = None
    state["gen"] += 1
    state["frame"] = None
    log.clear_output()
    readout.value = "Click a reply token after Generate."
    _show_ratios()
    _show_tokens()
    board.value = b""
    status.value = "Cleared. Generate draws the next training image."


def on_show_all(_change):
    _show_board()


def on_pick(change):
    raw = change["new"]
    if not raw or raw.count(",") != 2:
        return
    seq_s, gen_s, idx_s = raw.split(",")
    try:
        seq = int(seq_s)
        gen = int(gen_s)
        index = int(idx_s)
    except ValueError:
        return
    if seq <= state["ignore_through"] or state["busy"]:
        return
    if gen != state["gen"] or index < 0 or index >= len(tokens):
        return
    state["ignore_through"] = seq
    state["selected"] = index
    row = tokens[index]
    if row["mode"] == "mid":
        how = (
            "Midpoint token, through the ']' of [REMEMBER ...]. "
            "s and the green average are the agent. "
            "v and v_bar are not supervised."
        )
    else:
        how = "Full token. s, v, and v_bar are the stored label."
    readout.value = (
        f"question {row['q']} token {index} · {html.escape(row['mode'])}"
        f" · {html.escape(row['piece'])}<br>"
        f"{html.escape(how)}<br>"
        f"{html.escape(row['label_text'])}<br>"
        f"{_point_text('s', row['s'])} &nbsp; "
        f"{_point_text('v', row['v'])}<br>"
        f"{_point_text('v_bar', row['v_bar'])} &nbsp; "
        f"{_point_text('avg', row['avg'])}"
    )
    _show_tokens()
    _show_board()


begin_box = widgets.FloatText(
    value=BEGIN_FRACTION,
    description="Begin fraction:",
    step=0.05,
    style={"description_width": "110px"},
    layout=widgets.Layout(width="220px"),
)
gen_btn = widgets.Button(description="Generate", button_style="success", icon="play")
clear_btn = widgets.Button(description="Clear", icon="trash")
show_all = widgets.Checkbox(value=False, description="show all", indent=False)
gen_btn.on_click(on_generate)
clear_btn.on_click(on_clear)
show_all.observe(on_show_all, names="value")

ratios = widgets.HTML()
legend = widgets.HTML(
    "<span style='color:#1e5aff'>● s</span> &nbsp; "
    "<span style='color:#ff0000'>● v</span> &nbsp; "
    "<span style='color:#ff8c00'>● v_bar</span> &nbsp; "
    "<span style='color:#00c846'>● average of v and v_bar</span>"
)
readout = widgets.HTML("Click a reply token after Generate.")
status = widgets.HTML(
    "<span style='font-family:sans-serif;font-size:13px'>"
    "Generate draws one image from the localizer loop.</span>"
)
board = widgets.Image(format="png", width=BOARD_PX, height=BOARD_PX)
token_html = widgets.HTML()
log = widgets.Output(layout=widgets.Layout(
    border="1px solid #ccc",
    height="280px",
    max_height="280px",
    overflow_y="auto",
    width="100%",
))
log.add_class("dbg-log")
pick_box = widgets.Text(value="")
pick_box.add_class("dbg-pick")
pick_box.layout.height = "0px"
pick_box.layout.visibility = "hidden"
pick_box.layout.overflow = "hidden"
pick_box.observe(on_pick, names="value")

controls = (begin_box, gen_btn, clear_btn, show_all)
_show_ratios()

display(HTML(
    """
<style>
.dbg-tokens { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; }
.dbg-q { margin: 8px 0 12px 0; }
.dbg-h { font-weight: 600; margin-bottom: 2px; }
.dbg-line { white-space: pre-wrap; color: #333; }
.dbg-toks {
  white-space: pre-wrap;
  line-height: 1.8;
  border: 1px solid #ccc;
  padding: 8px;
  max-height: 160px;
  overflow-y: auto;
}
.dbg-tok { cursor: pointer; border-radius: 3px; }
.dbg-tok:hover { background: #e8eefc; }
.dbg-tok.dbg-on { background: #ffe08a; }
.dbg-mid { text-decoration: underline dotted #888; }
.dbg-eos { opacity: 0.55; }
.dbg-log {
  height: 280px !important;
  max-height: 280px !important;
  overflow-y: auto !important;
}
</style>
<script>
(function () {
  if (window.__dbgTokenClick) { return; }
  window.__dbgTokenClick = true;
  window.addEventListener("click", function (ev) {
    var span = ev.target && ev.target.closest && ev.target.closest(".dbg-tok");
    if (!span) { return; }
    var box = document.querySelector(".dbg-pick input");
    if (!box) { return; }
    window.__dbgPickN = (window.__dbgPickN || 0) + 1;
    var gen = span.getAttribute("data-gen") || "0";
    var i = span.getAttribute("data-i");
    box.value = window.__dbgPickN + "," + gen + "," + i;
    box.dispatchEvent(new Event("input", { bubbles: true }));
  }, true);
})();
</script>
"""
))
display(widgets.VBox([
    widgets.HBox([begin_box, gen_btn, clear_btn, show_all]),
    ratios,
    legend,
    readout,
    widgets.HBox([
        widgets.VBox([status, board], layout=widgets.Layout(flex="0 0 auto")),
        token_html,
    ], layout=widgets.Layout(width="100%", align_items="flex-start")),
    log,
    pick_box,
]))
